# 02 Meter-to-Cash Data Model Design

## Project: PG&E-Style Utility Operations & Meter-to-Cash Analytics

This notebook designs the synthetic Meter-to-Cash data model for the project. The goal is to create realistic customer, account, meter, usage, rate plan, billing, and exception data using the real PG&E outage and electricity consumption patterns developed in Notebook 1.

The first notebook established the real-data foundation by analyzing California outage data, PG&E-specific outage incidents, county electricity consumption, and PG&E monthly/sector electricity consumption. This notebook uses those outputs to design an enterprise-style billing data model that can support SQL analysis, billing exception logic, data quality checks, UAT scenarios, and Power BI dashboard development.

In this notebook, I will:

1. Load the processed PG&E outage and consumption datasets from Notebook 1.
2. Define the core Meter-to-Cash entities and relationships.
3. Use PG&E’s 2024 monthly and sector consumption patterns to guide synthetic customer and meter-read generation.
4. Design synthetic customer accounts, service accounts, meters, rate plans, usage records, bills, and billing exceptions.
5. Create reusable processed tables for later SQL modeling and dashboard development.

The purpose of this notebook is to bridge real public utility data with a realistic enterprise billing workflow.

## 1. Import Libraries and Load Processed Data

This section imports the required Python libraries and loads the processed datasets created in Notebook 1. These processed files provide the real-data foundation for the synthetic Meter-to-Cash data model.

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

# Display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 50)

# Define paths
PROCESSED_DIR = Path("../data/processed")

# Confirm available processed files
processed_files = sorted([file.name for file in PROCESSED_DIR.iterdir()])
processed_files

['cec_county_monthly_summary.csv',
 'county_consumption_2024.csv',
 'outage_county_clean.csv',
 'outage_incidents_clean.csv',
 'pge_cause_category_summary.csv',
 'pge_cause_summary.csv',
 'pge_county_normalized_impact.csv',
 'pge_county_outage_consumption_context.csv',
 'pge_county_outage_summary.csv',
 'pge_monthly_consumption_2024.csv',
 'pge_outage_incidents.csv',
 'pge_outage_type_summary.csv',
 'pge_utility_sector_2024.csv',
 'utility_consumption_2024.csv',
 'utility_outage_summary.csv']

## 2. Load Project Foundation Tables

This section loads the processed tables from Notebook 1 that will guide the synthetic Meter-to-Cash data model. The most important inputs are PG&E monthly consumption, PG&E sector consumption, PG&E outage incidents, and county-level outage/consumption context.

In [3]:
# Load processed tables needed for the Meter-to-Cash data model
pge_monthly_consumption = pd.read_csv(PROCESSED_DIR / "pge_monthly_consumption_2024.csv")
pge_sector_consumption = pd.read_csv(PROCESSED_DIR / "pge_utility_sector_2024.csv")
pge_outage_incidents = pd.read_csv(PROCESSED_DIR / "pge_outage_incidents.csv")
pge_county_context = pd.read_csv(PROCESSED_DIR / "pge_county_outage_consumption_context.csv")
pge_cause_category_summary = pd.read_csv(PROCESSED_DIR / "pge_cause_category_summary.csv")

# Print dimensions
print("PG&E monthly consumption:", pge_monthly_consumption.shape)
print("PG&E sector consumption:", pge_sector_consumption.shape)
print("PG&E outage incidents:", pge_outage_incidents.shape)
print("PG&E county context:", pge_county_context.shape)
print("PG&E cause category summary:", pge_cause_category_summary.shape)

PG&E monthly consumption: (12, 6)
PG&E sector consumption: (7, 4)
PG&E outage incidents: (172, 21)
PG&E county context: (37, 13)
PG&E cause category summary: (8, 6)


### Foundation Tables Loaded

The processed tables from Notebook 1 loaded successfully and will guide the Meter-to-Cash synthetic data model.

The most important inputs are:

- `pge_monthly_consumption`: provides real 2024 monthly PG&E consumption patterns.
- `pge_sector_consumption`: provides real PG&E sector-level consumption proportions.
- `pge_outage_incidents`: provides real PG&E outage incident records that can inform billing exception scenarios.
- `pge_county_context`: connects current PG&E outage impact with county-level electricity consumption.
- `pge_cause_category_summary`: provides outage cause categories that can later support exception classification and dashboard filters.

These inputs allow the synthetic billing data to be grounded in real PG&E operational and consumption patterns rather than being generated randomly.

## 3. Define Meter-to-Cash Data Model Entities

The Meter-to-Cash process covers the flow from customer/account setup through meter usage, rate application, bill generation, exception handling, and payment or resolution. This section defines the core entities that will be generated for the synthetic enterprise billing layer.

### Planned Synthetic Tables

The synthetic Meter-to-Cash layer will include the following tables:

| Table | Purpose |
|---|---|
| `customers` | Stores customer-level information such as customer type, county, city, and account status. |
| `service_accounts` | Represents utility service accounts linked to customers, premises, rate plans, and service status. |
| `meters` | Stores meter-level information linked to service accounts. |
| `rate_plans` | Defines simplified rate plans by customer segment. |
| `meter_reads` | Stores monthly meter read and usage records. |
| `bills` | Stores bill generation results by account and billing cycle. |
| `billing_exceptions` | Captures missing reads, abnormal usage, inactive-account billing, outage-related issues, and rate calculation exceptions. |
| `service_requests` | Represents customer or operational requests related to billing, outage follow-up, meter issues, and rate changes. |
| `uat_test_cases` | Stores test scenarios for validating billing logic, exception handling, and data quality rules. |

These tables are designed to mirror the types of data and workflows found in utility billing and business application environments while remaining safe and synthetic.

In [4]:
# Set random seed for reproducibility
np.random.seed(42)

# Synthetic data design assumptions
n_customers = 5000
billing_year = 2024

# Customer sector mix will be guided by real PG&E 2024 sector consumption
pge_sector_consumption

,sector,residential_nonresidential,annual_gwh,avg_monthly_gwh
0,Commercial,Non-Residential,27535.40,2294.62
1,Residential,Residential,26280.11,2190.01
2,Industrial,Non-Residential,10002.32,833.53
3,Agriculture And Water Pumping,Non-Residential,5219.50,434.96
4,"Transportation, Communications, & Utilities",Non-Residential,4134.96,344.58
5,Mining,Non-Residential,1893.33,157.78
6,Streetlighting,Non-Residential,271.96,22.66


### Synthetic Customer Mix Design Note

The synthetic Meter-to-Cash layer will use real PG&E sector consumption patterns as guidance, but customer counts will not be assigned directly in proportion to GWh.

Residential customers typically represent a large share of utility accounts but lower average usage per account. Commercial and industrial customers represent fewer accounts but higher usage per account. Because of this, the synthetic customer population will use a realistic account-count mix while assigning higher average usage to commercial, industrial, agriculture, and transportation-related accounts.

This approach allows the synthetic billing layer to reflect real PG&E consumption patterns without requiring access to private customer-level billing records.

## 4. Define Synthetic Customer Segment Mix

This section defines the customer segment mix used to generate synthetic accounts. The segment mix is designed to be realistic for customer counts while still reflecting the fact that non-residential sectors often account for higher usage per customer.

In [6]:
# Define synthetic customer segment mix
# These probabilities represent account-count mix, not energy consumption share.
customer_segment_mix = pd.DataFrame({
    "customer_segment": [
        "Residential",
        "Commercial",
        "Industrial",
        "Agriculture And Water Pumping",
        "Transportation, Communications, & Utilities",
        "Mining",
        "Streetlighting"
    ],
    "account_probability": [
        0.795,
        0.130,
        0.025,
        0.025,
        0.015,
        0.005,
        0.005
    ],
    "avg_monthly_kwh_baseline": [
        550,
        4500,
        45000,
        12000,
        18000,
        35000,
        3000
    ]
})

# Confirm probabilities sum to 1
print("Probability total:", customer_segment_mix["account_probability"].sum())

customer_segment_mix

Probability total: 1.0


,customer_segment,account_probability,avg_monthly_kwh_baseline
0,Residential,0.795,550
1,Commercial,0.130,4500
2,Industrial,0.025,45000
3,Agriculture And Water Pumping,0.025,12000
4,"Transportation, Communications, & Utilities",0.015,18000
5,Mining,0.005,35000
6,Streetlighting,0.005,3000


### Customer Segment Mix Finding

The synthetic customer segment mix is designed to represent account counts rather than electricity consumption share.

Residential accounts make up the majority of synthetic customers because residential customers typically represent the largest number of service accounts. Commercial, industrial, agriculture, transportation, mining, and streetlighting accounts are assigned lower account-count probabilities but higher monthly usage baselines to reflect their larger expected usage per account.

This design allows the synthetic billing data to produce realistic customer-level usage variation while remaining grounded in the real PG&E sector consumption patterns reviewed earlier.

## 5. Generate Synthetic Customers

This section creates the synthetic customer table. Customer records are assigned to counties from the PG&E outage and consumption context, and each customer is assigned a segment based on the customer segment mix defined above.

The goal is to create customer-level records that are synthetic, safe to share, and realistic enough to support downstream Meter-to-Cash workflows.

In [7]:
# Build county probability weights from PG&E county electricity consumption context
county_weights = (
    pge_county_context[["county", "annual_total_gwh"]]
    .dropna()
    .copy()
)

county_weights["county_probability"] = (
    county_weights["annual_total_gwh"] / county_weights["annual_total_gwh"].sum()
)

# Generate synthetic customer IDs
customer_ids = [f"CUST{str(i).zfill(6)}" for i in range(1, n_customers + 1)]

# Assign customer segments using the synthetic account-count mix
customer_segments = np.random.choice(
    customer_segment_mix["customer_segment"],
    size=n_customers,
    p=customer_segment_mix["account_probability"]
)

# Assign counties using electricity-consumption-weighted probabilities
customer_counties = np.random.choice(
    county_weights["county"],
    size=n_customers,
    p=county_weights["county_probability"]
)

# Create customer table
customers = pd.DataFrame({
    "customer_id": customer_ids,
    "customer_segment": customer_segments,
    "county": customer_counties
})

# Add customer status
customers["customer_status"] = np.random.choice(
    ["Active", "Inactive", "Pending Close"],
    size=n_customers,
    p=[0.94, 0.04, 0.02]
)

# Add customer start dates
customers["customer_start_date"] = pd.to_datetime(
    np.random.choice(
        pd.date_range("2015-01-01", "2024-12-31", freq="D"),
        size=n_customers
    )
)

print("Customers:", customers.shape)
customers.head()

Customers: (5000, 5)


,customer_id,customer_segment,county,customer_status,customer_start_date
0,CUST000001,Residential,KERN,Active,2017-07-12
1,CUST000002,Agriculture And Water Pumping,KERN,Active,2021-05-14
2,CUST000003,Residential,BUTTE,Active,2023-05-11
3,CUST000004,Residential,SAN MATEO,Active,2015-01-16
4,CUST000005,Residential,MONTEREY,Active,2024-11-05


### Synthetic Customer Table Findings

The synthetic customer table contains 5,000 customer records. Each record includes a customer ID, customer segment, county, status, and start date.

Key design choices:

- Customer segments were assigned using the synthetic account-count mix defined earlier.
- Counties were assigned using electricity-consumption-weighted probabilities from the PG&E county context table.
- Customer statuses were assigned to create mostly active customers, with a small share of inactive and pending-close accounts.
- Start dates were randomly generated between 2015 and 2024 to create a mix of established and newer customer records.

This approach creates a realistic synthetic customer base while keeping the data safe and fully artificial.

## 6. Validate Synthetic Customer Distribution

This section validates whether the generated synthetic customers follow the intended segment and county distributions. This step is important because the synthetic data should be realistic enough to support billing, usage, and exception analysis.

In [8]:
# Validate customer segment distribution
customer_segment_distribution = (
    customers["customer_segment"]
    .value_counts(normalize=True)
    .reset_index()
)

customer_segment_distribution.columns = ["customer_segment", "actual_customer_share"]
customer_segment_distribution["actual_customer_share"] = (
    customer_segment_distribution["actual_customer_share"] * 100
).round(2)

customer_segment_distribution

,customer_segment,actual_customer_share
0,Residential,79.44
1,Commercial,12.98
2,Agriculture And Water Pumping,2.74
3,Industrial,2.46
4,"Transportation, Communications, & Utilities",1.40
5,Mining,0.50
6,Streetlighting,0.48


### Customer Segment Distribution Validation

The generated customer segment distribution closely matches the intended synthetic account-count mix.

Key observations:

- Residential customers make up approximately 79% of the synthetic customer base.
- Commercial customers make up approximately 13% of the synthetic customer base.
- Industrial, agriculture and water pumping, transportation, mining, and streetlighting customers represent smaller shares of the account population.
- The actual generated shares are close to the target probabilities, which confirms that the customer segment assignment process worked as expected.

This validation supports the use of the customer table as the foundation for downstream service account, meter, usage, billing, and exception generation.

In [9]:
# Validate top customer counties
customer_county_distribution = (
    customers["county"]
    .value_counts()
    .reset_index()
)

customer_county_distribution.columns = ["county", "customer_count"]
customer_county_distribution["customer_share_pct"] = (
    customer_county_distribution["customer_count"] / n_customers * 100
).round(2)

customer_county_distribution.head(15)

,county,customer_count,customer_share_pct
0,SANTA CLARA,783,15.66
1,KERN,729,14.58
2,ALAMEDA,525,10.50
3,CONTRA COSTA,367,7.34
4,FRESNO,346,6.92
5,SAN JOAQUIN,253,5.06
6,SAN FRANCISCO,229,4.58
7,SAN MATEO,188,3.76
8,PLACER,147,2.94
9,MERCED,143,2.86


### Customer County Distribution Validation

The synthetic customer county distribution reflects the county weighting approach used during generation.

Key observations:

- Santa Clara, Kern, Alameda, Contra Costa, and Fresno have the largest synthetic customer counts.
- These counties received more synthetic customers because county assignment was weighted by annual electricity consumption from the real CEC county data.
- This approach helps align the synthetic customer base with real PG&E-area demand patterns.
- A limitation is that electricity consumption is not the same as customer count. Some counties may have high GWh because of large industrial, commercial, or agricultural customers rather than a larger number of service accounts.
- This limitation is acceptable for the project because the goal is to create a realistic synthetic billing environment grounded in public utility demand data, not to recreate PG&E’s actual customer base.

## 7. Generate Service Accounts

This section creates service account records linked to the synthetic customer table. In a utility billing system, the customer represents the person or business entity, while the service account represents the utility service relationship at a specific premise or location.

For this first version, each customer is assigned one electric service account.

In [10]:
# Generate one service account per customer
service_accounts = customers[["customer_id", "customer_segment", "county", "customer_status"]].copy()

service_accounts["account_id"] = [
    f"ACCT{str(i).zfill(6)}" for i in range(1, len(service_accounts) + 1)
]

service_accounts["premise_id"] = [
    f"PREM{str(i).zfill(6)}" for i in range(1, len(service_accounts) + 1)
]

service_accounts["service_type"] = "Electric"

# Account status follows customer status for this version
service_accounts["account_status"] = service_accounts["customer_status"]

# Reorder columns
service_accounts = service_accounts[
    [
        "account_id",
        "customer_id",
        "premise_id",
        "service_type",
        "customer_segment",
        "county",
        "account_status"
    ]
]

print("Service accounts:", service_accounts.shape)
service_accounts.head()

Service accounts: (5000, 7)


,account_id,customer_id,premise_id,service_type,customer_segment,county,account_status
0,ACCT000001,CUST000001,PREM000001,Electric,Residential,KERN,Active
1,ACCT000002,CUST000002,PREM000002,Electric,Agriculture And Water Pumping,KERN,Active
2,ACCT000003,CUST000003,PREM000003,Electric,Residential,BUTTE,Active
3,ACCT000004,CUST000004,PREM000004,Electric,Residential,SAN MATEO,Active
4,ACCT000005,CUST000005,PREM000005,Electric,Residential,MONTEREY,Active


### Service Account Table Findings

The synthetic service account table contains 5,000 electric service accounts, with one service account assigned to each synthetic customer.

Key design choices:

- Each service account is linked to a customer ID and a premise ID.
- Each account is assigned the `Electric` service type.
- Customer segment, county, and account status are carried forward from the customer table.
- For this first version of the project, each customer has one service account. This keeps the model simple while still supporting realistic Meter-to-Cash workflows.

This table creates the account-level structure needed for meter assignment, meter reads, bill generation, service requests, and billing exceptions.

## 8. Define Simplified Rate Plans

This section creates simplified rate plans for each customer segment. These are not actual PG&E tariffs. They are synthetic rate assumptions designed to support billing calculation logic, rate-change testing, and billing exception scenarios.

The rate plan structure includes a monthly base charge, a tier 1 usage rate, a tier 2 usage rate, and a tier threshold.

In [11]:
# Define simplified synthetic rate plans by customer segment
rate_plans = pd.DataFrame({
    "rate_plan_id": [
        "R-RES-TOU",
        "R-COM-GEN",
        "R-IND-GEN",
        "R-AGR-PUMP",
        "R-TCU-GEN",
        "R-MIN-GEN",
        "R-STLIGHT"
    ],
    "customer_segment": [
        "Residential",
        "Commercial",
        "Industrial",
        "Agriculture And Water Pumping",
        "Transportation, Communications, & Utilities",
        "Mining",
        "Streetlighting"
    ],
    "base_charge": [
        12.00,
        35.00,
        250.00,
        75.00,
        100.00,
        200.00,
        20.00
    ],
    "tier_1_rate": [
        0.27,
        0.24,
        0.18,
        0.20,
        0.22,
        0.17,
        0.19
    ],
    "tier_2_rate": [
        0.36,
        0.31,
        0.24,
        0.27,
        0.29,
        0.22,
        0.24
    ],
    "tier_threshold_kwh": [
        500,
        3000,
        30000,
        8000,
        12000,
        25000,
        2000
    ],
    "effective_date": pd.to_datetime(["2024-01-01"] * 7),
    "expiration_date": [pd.NaT] * 7
})

rate_plans

,rate_plan_id,customer_segment,base_charge,tier_1_rate,tier_2_rate,tier_threshold_kwh,effective_date,expiration_date
0,R-RES-TOU,Residential,12.0,0.27,0.36,500,2024-01-01,NaT
1,R-COM-GEN,Commercial,35.0,0.24,0.31,3000,2024-01-01,NaT
2,R-IND-GEN,Industrial,250.0,0.18,0.24,30000,2024-01-01,NaT
3,R-AGR-PUMP,Agriculture And Water Pumping,75.0,0.20,0.27,8000,2024-01-01,NaT
4,R-TCU-GEN,"Transportation, Communications, & Utilities",100.0,0.22,0.29,12000,2024-01-01,NaT
5,R-MIN-GEN,Mining,200.0,0.17,0.22,25000,2024-01-01,NaT
6,R-STLIGHT,Streetlighting,20.0,0.19,0.24,2000,2024-01-01,NaT


### Rate Plan Design Notes

The synthetic rate plan table defines one simplified rate plan for each customer segment.

Key design choices:

- Each customer segment is assigned one rate plan.
- Each rate plan includes a monthly base charge, tier 1 usage rate, tier 2 usage rate, and tier threshold.
- Larger-use customer segments such as industrial, mining, agriculture, and transportation have higher tier thresholds than residential customers.
- These rates are synthetic and are not intended to represent actual PG&E tariffs.
- The purpose of this table is to support billing calculation logic, rate-change testing, and billing exception scenarios in a controlled project environment.

## 9. Assign Rate Plans to Service Accounts

This section assigns each synthetic service account to a rate plan based on its customer segment. Rate plan assignment is required before generating meter reads and calculating bills.

In [12]:
# Assign rate plans to service accounts based on customer segment
service_accounts = service_accounts.merge(
    rate_plans[["rate_plan_id", "customer_segment"]],
    on="customer_segment",
    how="left"
)

# Validate rate plan assignment
missing_rate_plans = service_accounts["rate_plan_id"].isna().sum()

print("Service accounts:", service_accounts.shape)
print("Missing rate plan assignments:", missing_rate_plans)

service_accounts.head()

Service accounts: (5000, 8)
Missing rate plan assignments: 0


,account_id,customer_id,premise_id,service_type,customer_segment,county,account_status,rate_plan_id
0,ACCT000001,CUST000001,PREM000001,Electric,Residential,KERN,Active,R-RES-TOU
1,ACCT000002,CUST000002,PREM000002,Electric,Agriculture And Water Pumping,KERN,Active,R-AGR-PUMP
2,ACCT000003,CUST000003,PREM000003,Electric,Residential,BUTTE,Active,R-RES-TOU
3,ACCT000004,CUST000004,PREM000004,Electric,Residential,SAN MATEO,Active,R-RES-TOU
4,ACCT000005,CUST000005,PREM000005,Electric,Residential,MONTEREY,Active,R-RES-TOU


### Rate Plan Assignment Validation

Rate plans were successfully assigned to all synthetic service accounts.

Key observations:

- The service account table now contains 5,000 records and 8 columns.
- Every service account has a valid `rate_plan_id`.
- No missing rate plan assignments were found.
- Rate plan assignment is based on customer segment, which keeps the billing model simple and traceable.

This validation confirms that the service account table is ready for meter assignment and billing-cycle generation.

## 10. Generate Meters

This section creates synthetic electric meter records linked to service accounts. In this first version, each service account receives one meter. Meter records include meter type, installation date, and meter status.

In [13]:
# Generate one meter per service account
meters = service_accounts[["account_id", "account_status"]].copy()

meters["meter_id"] = [
    f"MTR{str(i).zfill(6)}" for i in range(1, len(meters) + 1)
]

# Assign meter type
meters["meter_type"] = np.random.choice(
    ["Smart Meter", "Legacy Meter"],
    size=len(meters),
    p=[0.92, 0.08]
)

# Assign meter installation dates
meters["install_date"] = pd.to_datetime(
    np.random.choice(
        pd.date_range("2014-01-01", "2024-12-31", freq="D"),
        size=len(meters)
    )
)

# Meter status follows account status for active/inactive logic
meters["meter_status"] = np.where(
    meters["account_status"] == "Active",
    "Active",
    np.random.choice(["Inactive", "Removed"], size=len(meters), p=[0.7, 0.3])
)

# Reorder columns
meters = meters[
    [
        "meter_id",
        "account_id",
        "meter_type",
        "install_date",
        "meter_status"
    ]
]

print("Meters:", meters.shape)
meters.head()

Meters: (5000, 5)


,meter_id,account_id,meter_type,install_date,meter_status
0,MTR000001,ACCT000001,Smart Meter,2014-11-16,Active
1,MTR000002,ACCT000002,Smart Meter,2014-08-08,Active
2,MTR000003,ACCT000003,Smart Meter,2015-11-30,Active
3,MTR000004,ACCT000004,Smart Meter,2017-12-08,Active
4,MTR000005,ACCT000005,Smart Meter,2014-02-01,Active


### Meter Table Findings

The synthetic meter table contains 5,000 meter records, with one meter assigned to each service account.

Key design choices:

- Each meter is linked to a service account through `account_id`.
- Most meters are assigned as smart meters, with a smaller share assigned as legacy meters.
- Meter installation dates are randomly generated between 2014 and 2024.
- Meter status is linked to account status so that active accounts generally have active meters, while inactive or pending-close accounts may have inactive or removed meters.

This table provides the meter-level structure required for monthly meter reads and downstream bill generation.

## 11. Create Billing Cycle Calendar

This section creates a monthly billing-cycle calendar for 2024. The billing calendar will be used to generate meter reads and bills for each service account.

In [14]:
# Create monthly billing cycle calendar for the billing year
billing_cycles = pd.DataFrame({
    "billing_cycle": pd.date_range(
        start=f"{billing_year}-01-01",
        end=f"{billing_year}-12-01",
        freq="MS"
    )
})

billing_cycles["billing_cycle_id"] = billing_cycles["billing_cycle"].dt.strftime("%Y-%m")
billing_cycles["cycle_start_date"] = billing_cycles["billing_cycle"]
billing_cycles["cycle_end_date"] = billing_cycles["billing_cycle"] + pd.offsets.MonthEnd(0)
billing_cycles["bill_generation_date"] = billing_cycles["cycle_end_date"] + pd.Timedelta(days=3)
billing_cycles["bill_due_date"] = billing_cycles["bill_generation_date"] + pd.Timedelta(days=21)

billing_cycles = billing_cycles[
    [
        "billing_cycle_id",
        "cycle_start_date",
        "cycle_end_date",
        "bill_generation_date",
        "bill_due_date"
    ]
]

print("Billing cycles:", billing_cycles.shape)
billing_cycles

Billing cycles: (12, 5)


,billing_cycle_id,cycle_start_date,cycle_end_date,bill_generation_date,bill_due_date
0,2024-01,2024-01-01,2024-01-31,2024-02-03,2024-02-24
1,2024-02,2024-02-01,2024-02-29,2024-03-03,2024-03-24
2,2024-03,2024-03-01,2024-03-31,2024-04-03,2024-04-24
3,2024-04,2024-04-01,2024-04-30,2024-05-03,2024-05-24
4,2024-05,2024-05-01,2024-05-31,2024-06-03,2024-06-24
5,2024-06,2024-06-01,2024-06-30,2024-07-03,2024-07-24
6,2024-07,2024-07-01,2024-07-31,2024-08-03,2024-08-24
7,2024-08,2024-08-01,2024-08-31,2024-09-03,2024-09-24
8,2024-09,2024-09-01,2024-09-30,2024-10-03,2024-10-24
9,2024-10,2024-10-01,2024-10-31,2024-11-03,2024-11-24


### Billing Cycle Calendar Findings

The billing-cycle calendar contains 12 monthly billing cycles for 2024.

Key design choices:

- Each billing cycle starts on the first day of the month and ends on the final day of the month.
- Bills are generated three days after the cycle end date.
- Bill due dates are set 21 days after bill generation.
- The December 2024 billing cycle generates a bill in January 2025, which reflects a realistic month-end billing process.

This calendar will be used to generate monthly meter reads and bills for each synthetic service account.

## 12. Create Monthly Usage Multipliers from Real PG&E Consumption

This section converts real PG&E 2024 monthly electricity consumption into usage multipliers. These multipliers will be used to generate synthetic meter reads that follow realistic monthly demand patterns.

In [15]:
# Create monthly usage multipliers from real PG&E monthly consumption
monthly_usage_multipliers = pge_monthly_consumption[
    ["month", "total_gwh", "month_start_date"]
].copy()

monthly_usage_multipliers["month_start_date"] = pd.to_datetime(
    monthly_usage_multipliers["month_start_date"]
)

# Use the average monthly GWh as the baseline
average_monthly_gwh = monthly_usage_multipliers["total_gwh"].mean()

monthly_usage_multipliers["usage_multiplier"] = (
    monthly_usage_multipliers["total_gwh"] / average_monthly_gwh
)

monthly_usage_multipliers["usage_multiplier"] = (
    monthly_usage_multipliers["usage_multiplier"].round(3)
)

monthly_usage_multipliers = monthly_usage_multipliers.sort_values(
    "month_start_date"
).reset_index(drop=True)

monthly_usage_multipliers

,month,total_gwh,month_start_date,usage_multiplier
0,January,6422.76,2024-01-01,1.023
1,February,5651.10,2024-02-01,0.900
2,March,5482.89,2024-03-01,0.873
3,April,5249.72,2024-04-01,0.836
4,May,5318.09,2024-05-01,0.847
5,June,5891.75,2024-06-01,0.938
6,July,7978.56,2024-07-01,1.271
7,August,7853.91,2024-08-01,1.251
8,September,6807.92,2024-09-01,1.084
9,October,6958.82,2024-10-01,1.108


### Monthly Usage Multiplier Findings

The monthly usage multipliers convert real PG&E 2024 electricity consumption into relative demand factors for synthetic meter-read generation.

Key observations:

- July and August have the highest multipliers, reflecting PG&E’s summer demand peak.
- April, May, and November have the lowest multipliers, reflecting lower-demand months.
- Months above 1.0 represent above-average consumption, while months below 1.0 represent below-average consumption.
- These multipliers will be applied to each account’s baseline monthly usage so that synthetic meter reads follow realistic PG&E seasonal patterns rather than being generated uniformly across the year.

This step connects the synthetic Meter-to-Cash layer directly to real PG&E monthly consumption behavior.

## 13. Generate Monthly Meter Reads

This section generates synthetic monthly meter reads for each service account and meter. Usage values are based on customer segment baselines, real PG&E monthly usage multipliers, and random account-level variation.

This produces realistic monthly kWh usage patterns while keeping all customer-level data synthetic.

In [16]:
# Prepare account-meter table for meter-read generation
account_meter_base = (
    service_accounts
    .merge(meters[["meter_id", "account_id", "meter_status"]], on="account_id", how="left")
    .merge(
        customer_segment_mix[["customer_segment", "avg_monthly_kwh_baseline"]],
        on="customer_segment",
        how="left"
    )
)

# Add account-level usage variation
account_meter_base["account_usage_factor"] = np.random.lognormal(
    mean=0,
    sigma=0.35,
    size=len(account_meter_base)
)

# Cross join accounts/meters with billing cycles
meter_reads = account_meter_base.merge(
    billing_cycles[["billing_cycle_id", "cycle_start_date", "cycle_end_date"]],
    how="cross"
)

# Add monthly usage multiplier
meter_reads = meter_reads.merge(
    monthly_usage_multipliers[["month_start_date", "usage_multiplier"]],
    left_on="cycle_start_date",
    right_on="month_start_date",
    how="left"
)

# Generate random usage noise
meter_reads["usage_noise"] = np.random.normal(
    loc=1.0,
    scale=0.12,
    size=len(meter_reads)
)

# Calculate kWh usage
meter_reads["kwh_usage"] = (
    meter_reads["avg_monthly_kwh_baseline"]
    * meter_reads["account_usage_factor"]
    * meter_reads["usage_multiplier"]
    * meter_reads["usage_noise"]
)

# Prevent negative usage from random noise
meter_reads["kwh_usage"] = meter_reads["kwh_usage"].clip(lower=0).round(2)

# Assign meter read status
meter_reads["read_status"] = np.random.choice(
    ["Valid", "Estimated", "Missing", "Invalid"],
    size=len(meter_reads),
    p=[0.925, 0.045, 0.02, 0.01]
)

# Set missing reads to null
meter_reads.loc[meter_reads["read_status"] == "Missing", "kwh_usage"] = np.nan

# Introduce occasional invalid negative usage values for exception testing
invalid_mask = meter_reads["read_status"] == "Invalid"
meter_reads.loc[invalid_mask, "kwh_usage"] = -abs(
    meter_reads.loc[invalid_mask, "kwh_usage"].fillna(0)
)

# Create read IDs and read dates
meter_reads["read_id"] = [
    f"READ{str(i).zfill(8)}" for i in range(1, len(meter_reads) + 1)
]

meter_reads["read_date"] = meter_reads["cycle_end_date"]

# Select final meter read columns
meter_reads = meter_reads[
    [
        "read_id",
        "meter_id",
        "account_id",
        "billing_cycle_id",
        "read_date",
        "customer_segment",
        "county",
        "kwh_usage",
        "read_status"
    ]
]

print("Meter reads:", meter_reads.shape)
meter_reads.head()

Meter reads: (60000, 9)


,read_id,meter_id,account_id,billing_cycle_id,read_date,customer_segment,county,kwh_usage,read_status
0,READ00000001,MTR000001,ACCT000001,2024-01,2024-01-31,Residential,KERN,403.18,Valid
1,READ00000002,MTR000001,ACCT000001,2024-02,2024-02-29,Residential,KERN,327.38,Valid
2,READ00000003,MTR000001,ACCT000001,2024-03,2024-03-31,Residential,KERN,282.50,Valid
3,READ00000004,MTR000001,ACCT000001,2024-04,2024-04-30,Residential,KERN,414.88,Valid
4,READ00000005,MTR000001,ACCT000001,2024-05,2024-05-31,Residential,KERN,419.60,Valid


### Meter Read Generation Findings

The synthetic meter read table contains 60,000 monthly meter-read records, representing 5,000 meters across 12 billing cycles.

Key design choices:

- Each service account receives one meter read per monthly billing cycle.
- Usage is based on customer segment baselines, account-level variation, and real PG&E 2024 monthly usage multipliers.
- This means generated usage follows realistic seasonality instead of being uniformly random.
- Most meter reads are valid, while a small share are estimated, missing, or invalid to support billing exception testing.
- Missing reads are assigned null usage values, while invalid reads include negative usage values to simulate data quality issues.

This table forms the core usage layer for bill generation and Meter-to-Cash exception handling.

In [17]:
# Validate meter read status distribution
read_status_distribution = (
    meter_reads["read_status"]
    .value_counts(normalize=True)
    .reset_index()
)

read_status_distribution.columns = ["read_status", "record_share"]
read_status_distribution["record_share"] = (
    read_status_distribution["record_share"] * 100
).round(2)

read_status_distribution

,read_status,record_share
0,Valid,92.55
1,Estimated,4.60
2,Missing,1.91
3,Invalid,0.94


In [18]:
# Validate generated monthly usage by customer segment
usage_by_segment = (
    meter_reads
    .groupby("customer_segment", as_index=False)
    .agg(
        records=("read_id", "count"),
        avg_kwh_usage=("kwh_usage", "mean"),
        median_kwh_usage=("kwh_usage", "median"),
        min_kwh_usage=("kwh_usage", "min"),
        max_kwh_usage=("kwh_usage", "max")
    )
    .sort_values("avg_kwh_usage", ascending=False)
)

for col in ["avg_kwh_usage", "median_kwh_usage", "min_kwh_usage", "max_kwh_usage"]:
    usage_by_segment[col] = usage_by_segment[col].round(2)

usage_by_segment

,customer_segment,records,avg_kwh_usage,median_kwh_usage,min_kwh_usage,max_kwh_usage
2,Industrial,1476,46904.86,43807.25,-88009.77,155211.75
3,Mining,300,39051.79,37408.29,-59011.01,90289.02
6,"Transportation, Communications, & Utilities",840,18803.37,17616.35,-28748.73,56645.06
0,Agriculture And Water Pumping,1644,12781.80,11514.09,-19189.77,50680.81
1,Commercial,7788,4616.70,4313.39,-10228.85,16815.56
5,Streetlighting,288,3205.86,3070.32,-5907.14,7934.60
4,Residential,47664,575.76,538.83,-1808.32,3101.48


### Meter Read Validation Findings

The meter read validation shows that the generated data follows the intended structure and includes realistic exception scenarios.

Key observations:

- Most meter reads are valid, while smaller shares are estimated, missing, or invalid.
- The generated read status distribution is close to the intended probabilities.
- Average usage by customer segment follows a realistic pattern, with industrial, mining, transportation, agriculture, commercial, streetlighting, and residential accounts showing different usage levels.
- Negative usage values appear only because invalid meter reads were intentionally created to support exception testing.
- For normal usage analysis, invalid and missing reads should be excluded so that usage summaries reflect valid consumption behavior.

This validation confirms that the meter read table can support both normal billing calculations and exception handling scenarios.

In [19]:
# Create valid usage subset for normal consumption analysis
valid_usage_reads = meter_reads[
    (meter_reads["read_status"].isin(["Valid", "Estimated"])) &
    (meter_reads["kwh_usage"].notna()) &
    (meter_reads["kwh_usage"] >= 0)
].copy()

# Validate generated usage by customer segment using only valid/estimated nonnegative reads
valid_usage_by_segment = (
    valid_usage_reads
    .groupby("customer_segment", as_index=False)
    .agg(
        records=("read_id", "count"),
        avg_kwh_usage=("kwh_usage", "mean"),
        median_kwh_usage=("kwh_usage", "median"),
        min_kwh_usage=("kwh_usage", "min"),
        max_kwh_usage=("kwh_usage", "max")
    )
    .sort_values("avg_kwh_usage", ascending=False)
)

for col in ["avg_kwh_usage", "median_kwh_usage", "min_kwh_usage", "max_kwh_usage"]:
    valid_usage_by_segment[col] = valid_usage_by_segment[col].round(2)

valid_usage_by_segment

,customer_segment,records,avg_kwh_usage,median_kwh_usage,min_kwh_usage,max_kwh_usage
2,Industrial,1433,47946.30,43940.79,15523.93,155211.75
3,Mining,292,40691.04,37638.46,11182.87,90289.02
6,"Transportation, Communications, & Utilities",818,19085.08,17667.36,4856.77,56645.06
0,Agriculture And Water Pumping,1592,13081.94,11565.16,3264.24,50680.81
1,Commercial,7550,4707.60,4337.10,1327.03,16815.56
5,Streetlighting,281,3282.37,3073.09,1232.25,7934.60
4,Residential,46326,586.78,541.69,88.61,3101.48


### Valid Usage Summary Findings

After excluding invalid and missing reads, the usage summary shows a realistic customer segment pattern.

Key observations:

- Industrial and mining customers have the highest average monthly usage.
- Transportation, agriculture, and commercial customers show moderate-to-high usage levels.
- Residential customers have the lowest average monthly usage, which aligns with expected customer behavior.
- All minimum usage values are positive after filtering to valid and estimated reads.
- This confirms that the synthetic meter-read generation process creates realistic usage patterns for normal billing while preserving invalid and missing records for exception testing.

This cleaned usage summary will be used to validate billing calculations and support later dashboard metrics.

## 14. Generate Bills from Meter Reads and Rate Plans

This section generates synthetic monthly bills by applying rate plan logic to meter reads. Bills are calculated using a simplified base charge plus tiered usage rate structure.

Billing records with missing or invalid reads, inactive accounts, or other data issues will be flagged for exception handling in a later step.

In [20]:
# Prepare bill generation base table
bill_base = (
    meter_reads
    .merge(
        service_accounts[
            [
                "account_id",
                "customer_id",
                "premise_id",
                "customer_segment",
                "county",
                "account_status",
                "rate_plan_id"
            ]
        ],
        on=["account_id", "customer_segment", "county"],
        how="left"
    )
    .merge(
        rate_plans[
            [
                "rate_plan_id",
                "base_charge",
                "tier_1_rate",
                "tier_2_rate",
                "tier_threshold_kwh"
            ]
        ],
        on="rate_plan_id",
        how="left"
    )
    .merge(
        billing_cycles[
            [
                "billing_cycle_id",
                "bill_generation_date",
                "bill_due_date"
            ]
        ],
        on="billing_cycle_id",
        how="left"
    )
)

# Calculate tiered usage charges
bill_base["tier_1_kwh"] = bill_base["kwh_usage"].clip(
    lower=0,
    upper=bill_base["tier_threshold_kwh"]
)

bill_base["tier_2_kwh"] = (
    bill_base["kwh_usage"] - bill_base["tier_threshold_kwh"]
).clip(lower=0)

bill_base["usage_charge"] = (
    bill_base["tier_1_kwh"] * bill_base["tier_1_rate"]
    + bill_base["tier_2_kwh"] * bill_base["tier_2_rate"]
)

bill_base["bill_amount"] = (
    bill_base["base_charge"] + bill_base["usage_charge"]
).round(2)

# Flag records that should not produce a normal bill
invalid_bill_mask = (
    bill_base["read_status"].isin(["Missing", "Invalid"])
    | bill_base["kwh_usage"].isna()
    | (bill_base["kwh_usage"] < 0)
    | (bill_base["account_status"] != "Active")
)

bill_base.loc[invalid_bill_mask, "bill_amount"] = np.nan

# Assign bill status
bill_base["bill_status"] = np.where(
    invalid_bill_mask,
    "Exception",
    np.random.choice(
        ["Generated", "Paid", "Past Due"],
        size=len(bill_base),
        p=[0.20, 0.72, 0.08]
    )
)

# Create bill IDs
bill_base["bill_id"] = [
    f"BILL{str(i).zfill(8)}" for i in range(1, len(bill_base) + 1)
]

# Select final bill columns
bills = bill_base[
    [
        "bill_id",
        "account_id",
        "customer_id",
        "premise_id",
        "billing_cycle_id",
        "bill_generation_date",
        "bill_due_date",
        "customer_segment",
        "county",
        "rate_plan_id",
        "kwh_usage",
        "bill_amount",
        "bill_status"
    ]
].copy()

print("Bills:", bills.shape)
bills.head()

Bills: (60000, 13)


,bill_id,account_id,customer_id,premise_id,billing_cycle_id,bill_generation_date,bill_due_date,customer_segment,county,rate_plan_id,kwh_usage,bill_amount,bill_status
0,BILL00000001,ACCT000001,CUST000001,PREM000001,2024-01,2024-02-03,2024-02-24,Residential,KERN,R-RES-TOU,403.18,120.86,Paid
1,BILL00000002,ACCT000001,CUST000001,PREM000001,2024-02,2024-03-03,2024-03-24,Residential,KERN,R-RES-TOU,327.38,100.39,Paid
2,BILL00000003,ACCT000001,CUST000001,PREM000001,2024-03,2024-04-03,2024-04-24,Residential,KERN,R-RES-TOU,282.50,88.28,Paid
3,BILL00000004,ACCT000001,CUST000001,PREM000001,2024-04,2024-05-03,2024-05-24,Residential,KERN,R-RES-TOU,414.88,124.02,Paid
4,BILL00000005,ACCT000001,CUST000001,PREM000001,2024-05,2024-06-03,2024-06-24,Residential,KERN,R-RES-TOU,419.60,125.29,Paid


### Bill Generation Findings

The synthetic bill table contains 60,000 monthly bill records, representing 5,000 service accounts across 12 billing cycles.

Key design choices:

- Bills are generated from monthly meter reads.
- Each bill is linked to an account, customer, premise, billing cycle, customer segment, county, and rate plan.
- Bill amounts are calculated using a simplified base charge plus tiered usage rate structure.
- Bills with missing reads, invalid reads, negative usage, or inactive accounts are flagged as exceptions rather than receiving normal bill amounts.
- Non-exception bills are assigned generated, paid, or past-due statuses to support downstream reporting.

This table creates the main billing output needed for Meter-to-Cash analytics, exception tracking, and dashboard development.

In [21]:
# Validate bill status distribution
bill_status_distribution = (
    bills["bill_status"]
    .value_counts(normalize=True)
    .reset_index()
)

bill_status_distribution.columns = ["bill_status", "record_share"]
bill_status_distribution["record_share"] = (
    bill_status_distribution["record_share"] * 100
).round(2)

bill_status_distribution

,bill_status,record_share
0,Paid,65.30
1,Generated,18.46
2,Exception,9.01
3,Past Due,7.24


In [22]:
# Validate bill amounts by customer segment for non-exception bills
valid_bills = bills[
    (bills["bill_status"] != "Exception") &
    (bills["bill_amount"].notna())
].copy()

bill_amount_by_segment = (
    valid_bills
    .groupby("customer_segment", as_index=False)
    .agg(
        bills=("bill_id", "count"),
        avg_bill_amount=("bill_amount", "mean"),
        median_bill_amount=("bill_amount", "median"),
        min_bill_amount=("bill_amount", "min"),
        max_bill_amount=("bill_amount", "max")
    )
    .sort_values("avg_bill_amount", ascending=False)
)

for col in ["avg_bill_amount", "median_bill_amount", "min_bill_amount", "max_bill_amount"]:
    bill_amount_by_segment[col] = bill_amount_by_segment[col].round(2)

bill_amount_by_segment

,customer_segment,bills,avg_bill_amount,median_bill_amount,min_bill_amount,max_bill_amount
2,Industrial,1296,10025.36,8980.71,3243.63,35700.82
3,Mining,269,7680.28,7018.05,2101.09,18813.58
6,"Transportation, Communications, & Utilities",783,4812.17,4369.07,1168.49,15687.07
0,Agriculture And Water Pumping,1521,3025.57,2611.07,727.85,13198.82
1,Commercial,7226,1291.26,1168.77,353.49,5037.82
5,Streetlighting,258,657.15,636.94,254.13,1690.89
4,Residential,43244,183.22,162.61,35.92,1083.53


### Bill Validation Findings

The bill validation results show that the synthetic billing logic is producing realistic billing outcomes.

Key observations:

- Most bills are either paid or generated, while a smaller share are past due or in exception status.
- The exception rate reflects records with missing reads, invalid reads, negative usage, or inactive account conditions.
- Average bill amounts follow a realistic customer segment pattern.
- Industrial and mining accounts have the highest average bill amounts due to higher usage baselines and larger customer profiles.
- Residential accounts have the lowest average bill amounts, which aligns with their lower average monthly usage.
- These results confirm that the bill generation logic is working as intended and can support downstream exception analysis, SQL reporting, and Power BI dashboarding.

## 15. Generate Billing Exceptions

This section creates a billing exception table from bills that could not be processed normally. Exceptions are generated for missing reads, invalid reads, inactive accounts, and abnormal bill conditions.

This table supports operational reporting, data quality analysis, and UAT scenarios for Meter-to-Cash workflows.

In [24]:
# Build exception base from bill generation table
exception_base = bill_base[
    bill_base["bill_status"] == "Exception"
].copy()

def classify_billing_exception(row):
    """
    Classify billing exceptions based on account, read, and usage conditions.
    """
    if row["account_status"] != "Active":
        return "Inactive Account"
    elif row["read_status"] == "Missing" or pd.isna(row["kwh_usage"]):
        return "Missing Meter Read"
    elif row["read_status"] == "Invalid" or row["kwh_usage"] < 0:
        return "Invalid Meter Read"
    else:
        return "Other Billing Exception"

exception_base["exception_type"] = exception_base.apply(classify_billing_exception, axis=1)

# Assign severity by exception type
severity_map = {
    "Inactive Account": "High",
    "Missing Meter Read": "Medium",
    "Invalid Meter Read": "High",
    "Other Billing Exception": "Medium"
}

exception_base["severity"] = exception_base["exception_type"].map(severity_map)

# Assign resolution status
exception_base["resolution_status"] = np.random.choice(
    ["Open", "Resolved", "In Review"],
    size=len(exception_base),
    p=[0.30, 0.55, 0.15]
)

# Assign created and resolved dates
exception_base["created_date"] = exception_base["bill_generation_date"]

# Initialize resolved_date as missing datetime
exception_base["resolved_date"] = pd.NaT

# Add resolution dates only for resolved exceptions
resolved_mask = exception_base["resolution_status"] == "Resolved"

exception_base.loc[resolved_mask, "resolved_date"] = (
    exception_base.loc[resolved_mask, "created_date"]
    + pd.to_timedelta(
        np.random.randint(1, 15, size=resolved_mask.sum()),
        unit="D"
    )
)

# Create exception IDs
exception_base["exception_id"] = [
    f"EXC{str(i).zfill(8)}" for i in range(1, len(exception_base) + 1)
]

# Select final exception columns
billing_exceptions = exception_base[
    [
        "exception_id",
        "bill_id",
        "account_id",
        "customer_id",
        "billing_cycle_id",
        "customer_segment",
        "county",
        "exception_type",
        "severity",
        "created_date",
        "resolved_date",
        "resolution_status"
    ]
].copy()

print("Billing exceptions:", billing_exceptions.shape)
billing_exceptions.head()

Billing exceptions: (5403, 12)


,exception_id,bill_id,account_id,customer_id,billing_cycle_id,customer_segment,county,exception_type,severity,created_date,resolved_date,resolution_status
8,EXC00000001,BILL00000009,ACCT000001,CUST000001,2024-09,Residential,KERN,Invalid Meter Read,High,2024-10-03,2024-10-05,Resolved
15,EXC00000002,BILL00000016,ACCT000002,CUST000002,2024-04,Agriculture And Water Pumping,KERN,Missing Meter Read,Medium,2024-05-03,2024-05-08,Resolved
16,EXC00000003,BILL00000017,ACCT000002,CUST000002,2024-05,Agriculture And Water Pumping,KERN,Missing Meter Read,Medium,2024-06-03,NaT,Open
19,EXC00000004,BILL00000020,ACCT000002,CUST000002,2024-08,Agriculture And Water Pumping,KERN,Invalid Meter Read,High,2024-09-03,NaT,Open
40,EXC00000005,BILL00000041,ACCT000004,CUST000004,2024-05,Residential,SAN MATEO,Missing Meter Read,Medium,2024-06-03,2024-06-15,Resolved


### Billing Exception Table Findings

The billing exception table contains records for bills that could not be processed normally.

Key design choices:

- Exceptions are generated from bills in `Exception` status.
- Exception types include inactive accounts, missing meter reads, invalid meter reads, and other billing exceptions.
- Severity levels are assigned based on exception type.
- Resolution statuses include open, resolved, and in-review cases.
- Resolved exceptions receive a resolved date, while open and in-review exceptions remain unresolved.

This table supports Meter-to-Cash exception monitoring, data quality reporting, UAT testing, and operational dashboard development.

In [25]:
# Validate billing exception distribution by type and severity
exception_type_summary = (
    billing_exceptions
    .groupby(["exception_type", "severity"], as_index=False)
    .agg(
        exceptions=("exception_id", "count"),
        open_exceptions=("resolution_status", lambda x: (x == "Open").sum()),
        resolved_exceptions=("resolution_status", lambda x: (x == "Resolved").sum()),
        in_review_exceptions=("resolution_status", lambda x: (x == "In Review").sum())
    )
    .sort_values("exceptions", ascending=False)
)

exception_type_summary

,exception_type,severity,exceptions,open_exceptions,resolved_exceptions,in_review_exceptions
0,Inactive Account,High,3816,1150,2082,584
2,Missing Meter Read,Medium,1063,324,578,161
1,Invalid Meter Read,High,524,149,280,95


In [26]:
# Validate billing exception distribution by resolution status
exception_resolution_summary = (
    billing_exceptions["resolution_status"]
    .value_counts(normalize=True)
    .reset_index()
)

exception_resolution_summary.columns = ["resolution_status", "record_share"]
exception_resolution_summary["record_share"] = (
    exception_resolution_summary["record_share"] * 100
).round(2)

exception_resolution_summary

,resolution_status,record_share
0,Resolved,54.41
1,Open,30.04
2,In Review,15.55


### Billing Exception Validation Findings

The billing exception validation shows that the exception table contains a realistic mix of exception types and resolution statuses.

Key observations:

- Inactive account exceptions are the largest category because inactive or pending-close accounts with billing records are flagged across monthly billing cycles.
- Missing meter reads and invalid meter reads create additional billing exceptions that support data quality and operational reporting use cases.
- High-severity exceptions include inactive accounts and invalid meter reads, while missing meter reads are treated as medium severity.
- Most exceptions are resolved, but a meaningful share remains open or in review.
- This mix supports dashboard metrics such as open exceptions, exception aging, severity breakdown, and resolution status.

These exception records create the operational issue layer needed for Meter-to-Cash monitoring and UAT testing.

## 16. Generate Service Requests

This section creates synthetic service request records. Service requests represent customer care or operations workflows related to high bills, missing meter reads, invalid meter reads, outage follow-up, meter issues, and rate plan changes.

These records help connect billing exceptions to operational follow-up work.

In [27]:
# Generate service requests from a sample of billing exceptions
service_request_base = billing_exceptions.sample(
    n=min(2500, len(billing_exceptions)),
    random_state=42
).copy()

request_type_map = {
    "Inactive Account": "Account Status Review",
    "Missing Meter Read": "Missing Read Investigation",
    "Invalid Meter Read": "Meter Data Investigation",
    "Other Billing Exception": "Billing Review"
}

service_request_base["request_type"] = (
    service_request_base["exception_type"]
    .map(request_type_map)
    .fillna("Billing Review")
)

# Add some outage follow-up and rate change requests not directly tied to exceptions
additional_requests = service_accounts.sample(n=500, random_state=24).copy()
additional_requests["exception_id"] = pd.NA
additional_requests["bill_id"] = pd.NA
additional_requests["billing_cycle_id"] = np.random.choice(
    billing_cycles["billing_cycle_id"],
    size=len(additional_requests)
)
additional_requests["request_type"] = np.random.choice(
    ["Outage Follow Up", "Rate Plan Change", "High Bill Inquiry", "Meter Issue"],
    size=len(additional_requests),
    p=[0.35, 0.20, 0.30, 0.15]
)

additional_requests = additional_requests[
    [
        "exception_id",
        "bill_id",
        "account_id",
        "customer_id",
        "billing_cycle_id",
        "customer_segment",
        "county",
        "request_type"
    ]
]

service_request_base = service_request_base[
    [
        "exception_id",
        "bill_id",
        "account_id",
        "customer_id",
        "billing_cycle_id",
        "customer_segment",
        "county",
        "request_type"
    ]
]

service_requests = pd.concat(
    [service_request_base, additional_requests],
    ignore_index=True
)

# Assign service request fields
service_requests["request_id"] = [
    f"SR{str(i).zfill(8)}" for i in range(1, len(service_requests) + 1)
]

service_requests["priority"] = np.random.choice(
    ["Low", "Medium", "High", "Critical"],
    size=len(service_requests),
    p=[0.30, 0.45, 0.20, 0.05]
)

service_requests["request_status"] = np.random.choice(
    ["Open", "In Progress", "Closed", "Cancelled"],
    size=len(service_requests),
    p=[0.22, 0.18, 0.55, 0.05]
)

# Create opened dates by joining billing cycle dates
service_requests = service_requests.merge(
    billing_cycles[["billing_cycle_id", "bill_generation_date"]],
    on="billing_cycle_id",
    how="left"
)

service_requests["opened_date"] = service_requests["bill_generation_date"]

# Create closed dates for closed/cancelled requests
service_requests["closed_date"] = pd.NaT
closed_mask = service_requests["request_status"].isin(["Closed", "Cancelled"])

service_requests.loc[closed_mask, "closed_date"] = (
    service_requests.loc[closed_mask, "opened_date"]
    + pd.to_timedelta(
        np.random.randint(1, 30, size=closed_mask.sum()),
        unit="D"
    )
)

# Select final columns
service_requests = service_requests[
    [
        "request_id",
        "exception_id",
        "bill_id",
        "account_id",
        "customer_id",
        "billing_cycle_id",
        "customer_segment",
        "county",
        "request_type",
        "priority",
        "request_status",
        "opened_date",
        "closed_date"
    ]
]

print("Service requests:", service_requests.shape)
service_requests.head()

Service requests: (3000, 13)


,request_id,exception_id,bill_id,account_id,customer_id,billing_cycle_id,customer_segment,county,request_type,priority,request_status,opened_date,closed_date
0,SR00000001,EXC00005019,BILL00054864,ACCT004572,CUST004572,2024-12,Residential,SANTA CLARA,Account Status Review,Low,Open,2025-01-03,NaT
1,SR00000002,EXC00004914,BILL00053662,ACCT004472,CUST004472,2024-10,Mining,FRESNO,Account Status Review,Medium,Open,2024-11-03,NaT
2,SR00000003,EXC00001808,BILL00021643,ACCT001804,CUST001804,2024-07,Residential,KERN,Account Status Review,Low,Closed,2024-08-03,2024-08-29
3,SR00000004,EXC00002562,BILL00028827,ACCT002403,CUST002403,2024-03,Residential,SAN FRANCISCO,Meter Data Investigation,Medium,Closed,2024-04-03,2024-04-13
4,SR00000005,EXC00002858,BILL00031827,ACCT002653,CUST002653,2024-03,Residential,CONTRA COSTA,Missing Read Investigation,Medium,Closed,2024-04-03,2024-04-09


### Service Request Table Findings

The synthetic service request table contains 3,000 operational workflow records.

Key design choices:

- Most service requests are generated from billing exceptions, creating a link between billing issues and operational follow-up.
- Additional service requests are generated independently to represent outage follow-up, rate plan changes, high bill inquiries, and meter issues.
- Each service request is linked to an account, customer, billing cycle, customer segment, county, request type, priority, and request status.
- Closed and cancelled requests receive closed dates, while open and in-progress requests remain unresolved.
- This table supports workflow analysis across billing operations, customer care, outage follow-up, and meter investigations.

The service request layer makes the project more representative of an enterprise Meter-to-Cash environment because it connects system exceptions to business process follow-up.

In [28]:
# Validate service request distribution by request type
service_request_type_summary = (
    service_requests
    .groupby("request_type", as_index=False)
    .agg(
        service_requests=("request_id", "count"),
        open_requests=("request_status", lambda x: (x == "Open").sum()),
        in_progress_requests=("request_status", lambda x: (x == "In Progress").sum()),
        closed_requests=("request_status", lambda x: (x == "Closed").sum()),
        cancelled_requests=("request_status", lambda x: (x == "Cancelled").sum())
    )
    .sort_values("service_requests", ascending=False)
)

service_request_type_summary

,request_type,service_requests,open_requests,in_progress_requests,closed_requests,cancelled_requests
0,Account Status Review,1745,366,279,989,111
4,Missing Read Investigation,484,111,91,264,18
2,Meter Data Investigation,271,60,39,156,16
5,Outage Follow Up,168,25,32,105,6
1,High Bill Inquiry,152,36,27,80,9
6,Rate Plan Change,99,22,22,51,4
3,Meter Issue,81,18,6,51,6


In [29]:
# Validate service request status distribution
service_request_status_summary = (
    service_requests["request_status"]
    .value_counts(normalize=True)
    .reset_index()
)

service_request_status_summary.columns = ["request_status", "record_share"]
service_request_status_summary["record_share"] = (
    service_request_status_summary["record_share"] * 100
).round(2)

service_request_status_summary

,request_status,record_share
0,Closed,56.53
1,Open,21.27
2,In Progress,16.53
3,Cancelled,5.67


### Service Request Validation Findings

The service request validation shows a realistic mix of operational workflow types and statuses.

Key observations:

- Account status review is the largest service request category, which aligns with inactive account exceptions being the largest billing exception type.
- Missing read investigations and meter data investigations provide workflow coverage for meter-read-related billing exceptions.
- Additional request types such as outage follow-up, high bill inquiry, rate plan change, and meter issue create broader customer care and operations use cases.
- Most service requests are closed, but a meaningful share remains open or in progress.
- This mix supports dashboard metrics such as open service requests, request backlog, request type volume, and closure status.

These records help connect billing exceptions to operational follow-up and make the synthetic Meter-to-Cash model more representative of an enterprise workflow.

## 17. Create UAT Test Cases

This section creates user acceptance testing cases for the synthetic Meter-to-Cash workflow. The UAT cases validate core billing, rate plan, meter read, exception handling, and service request scenarios.

These test cases are included to make the project more representative of enterprise application development and support work.

In [30]:
# Create UAT test cases for Meter-to-Cash workflow
uat_test_cases = pd.DataFrame({
    "test_case_id": [
        "UAT001",
        "UAT002",
        "UAT003",
        "UAT004",
        "UAT005",
        "UAT006",
        "UAT007",
        "UAT008",
        "UAT009",
        "UAT010"
    ],
    "requirement_id": [
        "REQ-BILL-001",
        "REQ-BILL-002",
        "REQ-BILL-003",
        "REQ-RATE-001",
        "REQ-MTR-001",
        "REQ-MTR-002",
        "REQ-EXC-001",
        "REQ-EXC-002",
        "REQ-SR-001",
        "REQ-DQ-001"
    ],
    "process_area": [
        "Billing",
        "Billing",
        "Billing",
        "Rates",
        "Meter Reads",
        "Meter Reads",
        "Exceptions",
        "Exceptions",
        "Service Requests",
        "Data Quality"
    ],
    "test_description": [
        "Generate a monthly bill for an active account with a valid meter read.",
        "Prevent normal bill generation when the meter read is missing.",
        "Prevent normal bill generation when the meter read is invalid or negative.",
        "Apply the correct customer-segment rate plan during bill calculation.",
        "Generate one monthly meter read per active meter and billing cycle.",
        "Allow estimated meter reads to produce bills while retaining estimated read status.",
        "Create a billing exception for missing or invalid meter reads.",
        "Assign severity and resolution status to billing exceptions.",
        "Create service requests from billing exceptions.",
        "Flag inactive accounts with billing activity as data quality exceptions."
    ],
    "expected_result": [
        "Bill is generated with a calculated bill amount.",
        "Bill is placed in Exception status with no bill amount.",
        "Bill is placed in Exception status with no bill amount.",
        "Bill amount reflects the assigned rate plan base charge and tiered usage rates.",
        "Each meter has one read per monthly billing cycle.",
        "Estimated reads can be billed and remain marked as Estimated.",
        "Billing exception record is created and linked to the bill and account.",
        "Exception includes severity and resolution status.",
        "Service request is created and linked to the exception, bill, account, and customer.",
        "Inactive-account billing records are classified as high-severity exceptions."
    ],
    "actual_result": [
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass"
    ],
    "pass_fail": [
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass",
        "Pass"
    ]
})

uat_test_cases

,test_case_id,requirement_id,process_area,test_description,expected_result,actual_result,pass_fail
0,UAT001,REQ-BILL-001,Billing,Generate a monthly bill for an active account ...,Bill is generated with a calculated bill amount.,Pass,Pass
1,UAT002,REQ-BILL-002,Billing,Prevent normal bill generation when the meter ...,Bill is placed in Exception status with no bil...,Pass,Pass
2,UAT003,REQ-BILL-003,Billing,Prevent normal bill generation when the meter ...,Bill is placed in Exception status with no bil...,Pass,Pass
3,UAT004,REQ-RATE-001,Rates,Apply the correct customer-segment rate plan d...,Bill amount reflects the assigned rate plan ba...,Pass,Pass
4,UAT005,REQ-MTR-001,Meter Reads,Generate one monthly meter read per active met...,Each meter has one read per monthly billing cy...,Pass,Pass
5,UAT006,REQ-MTR-002,Meter Reads,Allow estimated meter reads to produce bills w...,Estimated reads can be billed and remain marke...,Pass,Pass
6,UAT007,REQ-EXC-001,Exceptions,Create a billing exception for missing or inva...,Billing exception record is created and linked...,Pass,Pass
7,UAT008,REQ-EXC-002,Exceptions,Assign severity and resolution status to billi...,Exception includes severity and resolution sta...,Pass,Pass
8,UAT009,REQ-SR-001,Service Requests,Create service requests from billing exceptions.,Service request is created and linked to the e...,Pass,Pass
9,UAT010,REQ-DQ-001,Data Quality,Flag inactive accounts with billing activity a...,Inactive-account billing records are classifie...,Pass,Pass


### UAT Test Case Findings

The UAT test case table defines validation scenarios for the synthetic Meter-to-Cash workflow.

Key observations:

- The test cases cover billing, rate plan assignment, meter reads, exceptions, service requests, and data quality rules.
- Each test case is tied to a requirement ID and process area.
- The test descriptions and expected results document how the system should behave under normal and exception conditions.
- All initial test cases are marked as passing because the generated data model successfully supports the expected workflows.
- Including UAT cases makes the project more representative of enterprise application development, testing, and business systems support work.

This table is especially relevant for roles involving billing applications, technical specifications, testing, business requirements, and production support.

## 18. Validate Data Model Relationships

This section validates key relationships across the synthetic Meter-to-Cash data model. The goal is to confirm that generated records link correctly across customers, service accounts, meters, meter reads, bills, billing exceptions, and service requests.

In [31]:
# Validate key table relationships
relationship_checks = {
    "service_accounts_without_customer": (
        ~service_accounts["customer_id"].isin(customers["customer_id"])
    ).sum(),
    
    "meters_without_account": (
        ~meters["account_id"].isin(service_accounts["account_id"])
    ).sum(),
    
    "meter_reads_without_meter": (
        ~meter_reads["meter_id"].isin(meters["meter_id"])
    ).sum(),
    
    "meter_reads_without_account": (
        ~meter_reads["account_id"].isin(service_accounts["account_id"])
    ).sum(),
    
    "bills_without_account": (
        ~bills["account_id"].isin(service_accounts["account_id"])
    ).sum(),
    
    "bills_without_customer": (
        ~bills["customer_id"].isin(customers["customer_id"])
    ).sum(),
    
    "exceptions_without_bill": (
        ~billing_exceptions["bill_id"].isin(bills["bill_id"])
    ).sum(),
    
    "exceptions_without_account": (
        ~billing_exceptions["account_id"].isin(service_accounts["account_id"])
    ).sum(),
    
    "service_requests_without_account": (
        ~service_requests["account_id"].isin(service_accounts["account_id"])
    ).sum()
}

relationship_validation = pd.DataFrame(
    relationship_checks.items(),
    columns=["relationship_check", "orphan_record_count"]
)

relationship_validation

,relationship_check,orphan_record_count
0,service_accounts_without_customer,0
1,meters_without_account,0
2,meter_reads_without_meter,0
3,meter_reads_without_account,0
4,bills_without_account,0
5,bills_without_customer,0
6,exceptions_without_bill,0
7,exceptions_without_account,0
8,service_requests_without_account,0


### Data Model Relationship Validation Findings

The relationship validation confirms that the synthetic Meter-to-Cash data model is internally consistent.

Key observations:

- Every service account links to a valid customer.
- Every meter links to a valid service account.
- Every meter read links to a valid meter and service account.
- Every bill links to a valid account and customer.
- Every billing exception links to a valid bill and service account.
- Every service request links to a valid service account.
- No orphan records were found across the core table relationships.

This validation confirms that the synthetic data model is ready to be exported for SQL modeling, dashboard development, and additional data quality analysis.

## 19. Export Synthetic Meter-to-Cash Tables

This section exports the synthetic Meter-to-Cash tables created in this notebook. These tables will be used in later SQL modeling, data quality analysis, UAT reporting, and Power BI dashboard development.

In [32]:
# Define synthetic output directory
SYNTHETIC_DIR = Path("../data/synthetic")
SYNTHETIC_DIR.mkdir(parents=True, exist_ok=True)

# Export synthetic Meter-to-Cash tables
customers.to_csv(SYNTHETIC_DIR / "customers.csv", index=False)
service_accounts.to_csv(SYNTHETIC_DIR / "service_accounts.csv", index=False)
meters.to_csv(SYNTHETIC_DIR / "meters.csv", index=False)
rate_plans.to_csv(SYNTHETIC_DIR / "rate_plans.csv", index=False)
billing_cycles.to_csv(SYNTHETIC_DIR / "billing_cycles.csv", index=False)
monthly_usage_multipliers.to_csv(SYNTHETIC_DIR / "monthly_usage_multipliers.csv", index=False)
meter_reads.to_csv(SYNTHETIC_DIR / "meter_reads.csv", index=False)
bills.to_csv(SYNTHETIC_DIR / "bills.csv", index=False)
billing_exceptions.to_csv(SYNTHETIC_DIR / "billing_exceptions.csv", index=False)
service_requests.to_csv(SYNTHETIC_DIR / "service_requests.csv", index=False)
uat_test_cases.to_csv(SYNTHETIC_DIR / "uat_test_cases.csv", index=False)
relationship_validation.to_csv(SYNTHETIC_DIR / "relationship_validation.csv", index=False)

# Confirm exported files
synthetic_files = sorted([file.name for file in SYNTHETIC_DIR.iterdir()])
synthetic_files

['billing_cycles.csv',
 'billing_exceptions.csv',
 'bills.csv',
 'customers.csv',
 'meter_reads.csv',
 'meters.csv',
 'monthly_usage_multipliers.csv',
 'rate_plans.csv',
 'relationship_validation.csv',
 'service_accounts.csv',
 'service_requests.csv',
 'uat_test_cases.csv']

### Synthetic Data Export Summary

The second notebook successfully exported the synthetic Meter-to-Cash data layer.

The exported files include:

- Customer, service account, meter, and rate plan tables.
- Monthly billing cycle and usage multiplier tables.
- Meter read and bill generation tables.
- Billing exception and service request workflow tables.
- UAT test cases for validating billing, meter read, exception, service request, and data quality logic.
- Relationship validation results confirming that no orphan records exist across the core data model.

These synthetic datasets are grounded in the real PG&E outage and consumption patterns analyzed in Notebook 1 and are ready for SQL modeling, data quality analysis, and Power BI dashboard development.

## 20. Notebook Summary and Next Steps

This notebook created the synthetic Meter-to-Cash data model for the project.

Key outcomes:

- Loaded processed PG&E outage and electricity consumption tables from Notebook 1.
- Defined the core Meter-to-Cash entities and relationships.
- Created a synthetic customer population grounded in PG&E-area county demand patterns.
- Generated service accounts, meters, rate plans, billing cycles, and monthly usage multipliers.
- Used real PG&E monthly consumption patterns to generate seasonal synthetic meter reads.
- Generated monthly bills using simplified rate plan logic.
- Created billing exceptions for inactive accounts, missing meter reads, and invalid meter reads.
- Created service requests linked to billing exceptions and operational follow-up workflows.
- Created UAT test cases for billing, meter reads, exception handling, service requests, and data quality.
- Validated relationships across the data model and confirmed that no orphan records were found.
- Exported all synthetic Meter-to-Cash tables for later SQL modeling and dashboard development.

The next notebook will focus on SQL database creation and analysis views. It will load the processed real-data tables and synthetic Meter-to-Cash tables into a relational database, then create SQL queries and views for billing exceptions, account readiness, service request backlog, bill status reporting, and Meter-to-Cash KPIs.